
# Sổ tay Machine Learning (tiếng Việt) — từ bài giảng đến thực hành

Notebook này bám sát các bài giảng bạn đã gửi: khái niệm chung, hồi quy tuyến tính, phân lớp (Naive Bayes, Logistic), cây quyết định, SVM, kèm quy trình đánh giá & chọn mô hình.  
**Yêu cầu môi trường:** `python>=3.9`, `numpy`, `pandas`, `matplotlib`, `scikit-learn`.


In [ ]:

# (Tuỳ chọn) Cài đặt thư viện nếu thiếu — bỏ dấu # để chạy trong môi trường của bạn
# !pip install -U numpy pandas matplotlib scikit-learn



## 0) Import & tiện ích chung


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.datasets import make_regression, load_iris, load_wine, load_breast_cancer, make_moons
from sklearn.linear_model import LinearRegression, SGDRegressor, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC

np.random.seed(42)



## 1) Khung & quy trình làm ML
**Mục tiêu:** Tổng quát hoá tốt trên dữ liệu chưa thấy.  
**Quy trình:** Xác định bài toán → Chuẩn bị dữ liệu/đặc trưng → Chọn mô hình → Huấn luyện/điều chỉnh siêu tham số → Đánh giá (validation/test) → Triển khai.

Chúng ta sẽ dùng `train_test_split`, `KFold`/`cross_val_score`, và `GridSearchCV` để đảm bảo đánh giá đúng.


In [ ]:

# Ví dụ khởi tạo KFold cho 5-fold CV (minh hoạ)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
print(kfold)



## 2) Hồi quy tuyến tính — so sánh tiếp cận
- **Normal Equation** (nghiệm đóng) qua `LinearRegression` của scikit-learn.
- **Gradient Descent** (lặp) qua `SGDRegressor` hoặc code tay tối giản.

Ta dùng dữ liệu tổng hợp `make_regression`.


In [ ]:

# Tạo dữ liệu hồi quy tổng hợp
X_reg, y_reg = make_regression(n_samples=500, n_features=3, noise=15.0, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 2.1 Normal Equation (LinearRegression)
lin = LinearRegression()
lin.fit(Xr_train, yr_train)
pred_test_lin = lin.predict(Xr_test)
mse_lin = metrics.mean_squared_error(yr_test, pred_test_lin)
r2_lin = metrics.r2_score(yr_test, pred_test_lin)

print("LinearRegression (nghiệm đóng) — MSE:", mse_lin, "| R2:", r2_lin)

# 2.2 Gradient Descent (SGDRegressor)
sgd = SGDRegressor(max_iter=2000, tol=1e-4, random_state=42, learning_rate="invscaling", eta0=0.1)
sgd.fit(Xr_train, yr_train)
pred_test_sgd = sgd.predict(Xr_test)
mse_sgd = metrics.mean_squared_error(yr_test, pred_test_sgd)
r2_sgd = metrics.r2_score(yr_test, pred_test_sgd)
print("SGDRegressor (GD)        — MSE:", mse_sgd, "| R2:", r2_sgd)


In [ ]:

# Vẽ scatter dự đoán vs. thực tế (mỗi mô hình một plot riêng)
fig = plt.figure()
plt.scatter(yr_test, pred_test_lin)
plt.xlabel("Giá trị thật (y)")
plt.ylabel("Dự đoán (LinearRegression)")
plt.title("Hồi quy: Dự đoán vs Thực tế (LinearRegression)")
plt.show()

fig = plt.figure()
plt.scatter(yr_test, pred_test_sgd)
plt.xlabel("Giá trị thật (y)")
plt.ylabel("Dự đoán (SGDRegressor)")
plt.title("Hồi quy: Dự đoán vs Thực tế (SGDRegressor)")
plt.show()



## 3) Phân lớp (P1) — Naive Bayes & Logistic Regression
- **Naive Bayes (GaussianNB)**: tiếp cận sinh (generative), ước lượng \(p(x|y)\), suy ra \(p(y|x)\).
- **Logistic Regression**: tiếp cận phân biệt (discriminative), mô hình hoá trực tiếp ranh giới quyết định.

Dùng bộ **Iris** và **Breast Cancer** (tích hợp trong scikit-learn, không cần tải Internet).


In [ ]:

# Naive Bayes trên Iris
iris = load_iris()
Xi, yi = iris.data, iris.target
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2, random_state=42, stratify=yi)

nb = GaussianNB()
nb.fit(Xi_tr, yi_tr)
pred_nb = nb.predict(Xi_te)

print("GaussianNB — accuracy:", metrics.accuracy_score(yi_te, pred_nb))
print(metrics.classification_report(yi_te, pred_nb, target_names=iris.target_names))


In [ ]:

# Logistic Regression trên Breast Cancer (chuẩn hoá để hội tụ tốt)
bc = load_breast_cancer()
Xb, yb = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=42, stratify=yb)

pipe_log = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=2000, random_state=42))
])
pipe_log.fit(Xb_tr, yb_tr)
pred_log = pipe_log.predict(Xb_te)

print("LogisticRegression — accuracy:", metrics.accuracy_score(yb_te, pred_log))
print(metrics.classification_report(yb_te, pred_log, target_names=bc.target_names))



## 4) Phân lớp (P2) — Cây quyết định (Decision Tree)
Ý tưởng: tách dữ liệu theo điều kiện trên thuộc tính để tăng độ “thuần” của nhãn. Ta sẽ thử trên **Wine** và xem **feature importances**.


In [ ]:

wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=42, stratify=yw)

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(Xw_tr, yw_tr)
pred_tree = tree.predict(Xw_te)

print("DecisionTree — accuracy:", metrics.accuracy_score(yw_te, pred_tree))

# Top thuộc tính quan trọng
importances = pd.Series(tree.feature_importances_, index=wine.feature_names).sort_values(ascending=False)
print(importances.head(10))


In [ ]:

# Vẽ cây ở mức độ tổng quát (một plot riêng)
fig = plt.figure(figsize=(10, 6))
plot_tree(tree, feature_names=wine.feature_names, class_names=[str(c) for c in np.unique(yw)], filled=False)
plt.title("Cây quyết định (độ sâu=4) — dữ liệu Wine")
plt.show()



## 5) Phân lớp (P3) — SVM (biên tối ưu), hạt nhân RBF
Dùng dữ liệu **make_moons** (không tuyến tính), so sánh tham số `C` và `gamma`. Chúng ta cũng minh hoạ **biên quyết định**.


In [ ]:

Xm, ym = make_moons(n_samples=500, noise=0.25, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=42, stratify=ym)

def plot_decision_boundary(model, X, y, title=""):
    # Lưới để vẽ biên
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    fig = plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

# Hai cấu hình để so sánh (mỗi cấu hình 1 plot)
svm1 = Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))])
svm1.fit(Xm_tr, ym_tr)
acc1 = metrics.accuracy_score(ym_te, svm1.predict(Xm_te))
plot_decision_boundary(svm1, Xm_te, ym_te, title=f"SVM RBF: C=1.0, gamma='scale' — acc={acc1:.3f}")

svm2 = Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="rbf", C=50.0, gamma=0.5, random_state=42))])
svm2.fit(Xm_tr, ym_tr)
acc2 = metrics.accuracy_score(ym_te, svm2.predict(Xm_te))
plot_decision_boundary(svm2, Xm_te, ym_te, title=f"SVM RBF: C=50.0, gamma=0.5 — acc={acc2:.3f}")



## 6) Chọn mô hình & siêu tham số — GridSearchCV
Ta dùng lưới cho SVM RBF trên `make_moons`. Kết quả gồm tham số tốt nhất và điểm cross-validation trung bình.


In [ ]:

pipe_svm = Pipeline([("scaler", StandardScaler()), ("svc", SVC(kernel="rbf", random_state=42))])
param_grid = {
    "svc__C": [0.1, 1, 10, 50],
    "svc__gamma": ["scale", 0.1, 0.5, 1.0]
}
gs = GridSearchCV(pipe_svm, param_grid=param_grid, cv=5, n_jobs=None, scoring="accuracy")
gs.fit(Xm_tr, ym_tr)
print("Best params:", gs.best_params_)
print("Best CV score:", gs.best_score_)

best = gs.best_estimator_
test_acc = metrics.accuracy_score(ym_te, best.predict(Xm_te))
print("Test accuracy với mô hình tốt nhất:", test_acc)



## 7) Báo cáo nhanh & lưu mô hình
Minh hoạ với mô hình tốt nhất ở trên.


In [ ]:

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import joblib

# Ma trận nhầm lẫn (một plot riêng)
cm = confusion_matrix(ym_te, best.predict(Xm_te))
fig = plt.figure()
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Ma trận nhầm lẫn — SVM tốt nhất")
plt.show()

# Classification report
print(classification_report(ym_te, best.predict(Xm_te)))

# Lưu mô hình
joblib.dump(best, "/mnt/data/svm_best_model.joblib")
print("Đã lưu mô hình vào /mnt/data/svm_best_model.joblib")



## 8) Bài tập gợi ý
1. Thử **PolynomialFeatures** + LinearRegression cho hồi quy; so sánh mức độ overfitting bằng learning curve.
2. Thử **DecisionTreeClassifier** với `max_depth` khác nhau; quan sát accuracy & kích thước cây.
3. Với **LogisticRegression**, so sánh giữa `C` nhỏ/lớn (regularization mạnh/yếu).
4. Với **SVM**, thêm các giá trị `C`/`gamma` khác để thấy ranh giới quyết định thay đổi ra sao.
5. Tự viết **gradient descent** cho hồi quy tuyến tính (phiên bản code tay) và so sánh với `SGDRegressor`.
